In [1]:
import chipwhisperer as cw
import os, time

#bitstream_path =  r'C:\Users\sbista\ChipWhisperer\chipwhisperer\firmware\fpgas\aes\vivado_\cw305_aes.runs\impl_100t\cw305_top.bit'
bitstream_path =  r'/home/sareeta/chipwhisperer/firmware/fpgas/aes/vivado/cw305_aes.runs/impl_100t/cw305_top.bit'
assert os.path.isfile(bitstream_path), f"Bitstream not found: {bitstream_path}"

# 2) Connect to the capture board (CWLite)
scope = cw.scope()
scope.default_setup()

# 3) Connect and Program the FPGA
print("Programming CW305 FPGA with:", bitstream_path)
target = cw.target(scope, cw.targets.CW305, bsfile=bitstream_path, force=True) 

(ChipWhisperer NAEUSB WARNING|File naeusb.py:826) Your firmware (0.51.0) is outdated - latest is 0.54.0 See https://chipwhisperer.readthedocs.io/en/latest/firmware.html for more information


scope.gain.mode                          changed from low                       to high                     
scope.gain.gain                          changed from 0                         to 30                       
scope.gain.db                            changed from 5.5                       to 24.8359375               
scope.adc.basic_mode                     changed from low                       to rising_edge              
scope.adc.samples                        changed from 24400                     to 5000                     
scope.adc.trig_count                     changed from 36458043                  to 47480231                 
scope.clock.adc_src                      changed from clkgen_x1                 to clkgen_x4                
scope.clock.adc_freq                     changed from 987419                    to 29538459                 
scope.clock.adc_rate                     changed from 987419.0                  to 29538459.0               
scope.clock.clkgen_

(ChipWhisperer Target WARNING|File CW305.py:591) Using default Verilog defines (/home/sareeta/chipwhisperer/software/chipwhisperer/hardware/firmware/cw305/cw305_aes_defines.v); if this is not what you want, provide them via the defines_files argument


In [6]:
from Crypto.Cipher import AES

# Known NIST AES-128 test vector
KEY = bytes.fromhex("2b7e151628aed2a6abf7158809cf4f3c")
PT  = bytes.fromhex("5c692f9103b2302914d7e555e4dcee49")

# Expected ciphertext
EXP_CT = AES.new(KEY, AES.MODE_ECB).encrypt(PT)
print("Expected CT:", EXP_CT.hex())
# Write KEY
target.fpga_write(target.REG_CRYPT_KEY, KEY)

# Write PLAINTEXT
target.fpga_write(target.REG_CRYPT_TEXTIN, PT)

# Trigger encryption
target.fpga_write(target.REG_CRYPT_GO, b"\x01")

# Small delay (AES is fast but be safe)
time.sleep(0.01)
# Read ciphertext
ct = bytes(target.fpga_read(target.REG_CRYPT_CIPHEROUT, 16))
print("FPGA CT   :", ct.hex())
print("MATCH?    :", ct == EXP_CT)

# Set target round to 10 (the final round)
target.fpga_write(0x0C, [8]) 

# Run AES
target.fpga_write(target.REG_CRYPT_GO, [0x01])
time.sleep(0.01)

# Read both
ct = target.fpga_read(target.REG_CRYPT_CIPHEROUT, 16)
snap = target.fpga_read(0x0E, 16)

print(f"Cipherout: {ct.hex().upper()}")
print(f"Snapshot : {snap.hex().upper()}")

scope.arm()
target.fpga_write(target.REG_CRYPT_TEXTIN, b"\x00"*16)
target.fpga_write(target.REG_CRYPT_GO, b"\x01")

if scope.capture():
    print("❌ Trigger not seen")
else:
    print("✅ Trigger OK")

Expected CT: 06f36a65e8a99ff8907b2e5e5ddd77de
FPGA CT   : 06f36a65e8a99ff8907b2e5e5ddd77de
MATCH?    : True
Cipherout: 06F36A65E8A99FF8907B2E5E5DDD77DE
Snapshot : C3EF9D9E3908C4D3B50BF2948B08FE07
✅ Trigger OK


In [7]:
import chipwhisperer as cw

# ── Glitch module setup ────────────────────────────────────────
scope.glitch.clk_src      = "clkgen"       # sync to CW305 clock
scope.glitch.output       = "enable_only"  # output a pulse, no VCC glitch
scope.glitch.trigger_src  = "ext_single"   # fire once per external trigger
scope.glitch.repeat       = 1              # single pulse

# ── Start with delay = 0, we will sweep this later ────────────
scope.glitch.ext_offset   = 0    # cycles AFTER trigger before pulse fires
scope.glitch.width        = 5    # pulse width in glitch cycles (~5 cycles)

print("Glitch module configured as trigger output")
print(f"  clk_src     : {scope.glitch.clk_src}")
print(f"  output      : {scope.glitch.output}")
print(f"  trigger_src : {scope.glitch.trigger_src}")
print(f"  ext_offset  : {scope.glitch.ext_offset}")
print(f"  width       : {scope.glitch.width}")

Glitch module configured as trigger output
  clk_src     : clkgen
  output      : enable_only
  trigger_src : ext_single
  ext_offset  : 0
  width       : 5.078125


In [19]:
from chipshouter import ChipSHOUTER

# ── Connect ChipShouter ───────────────────────────────────────
# Windows → "COM3", "COM4" etc.
# Linux   → "/dev/ttyUSB0" or "/dev/ttyACM0"
chs = ChipSHOUTER("/dev/ttyUSB0")

# ── Safety-first starting parameters ─────────────────────────
chs.voltage     = 150      # Volts — start LOW, increase later
chs.pulse.width = 100      # nanoseconds
#CS.repeat      = 1        # single shot per trigger

# ── Arm it ───────────────────────────────────────────────────
chs.armed = True

print(f"ChipShouter armed : {cs.armed}")
print(f"Voltage           : {cs.voltage}V")
print(f"Pulse width       : {cs.pulse.width}ns")

ChipShouter armed : False
Voltage           : set      = 150
measured = 23
V
Pulse width       : 100ns


In [20]:
def verify_trigger_chain(scope, target):
    """
    Dry run — confirms CW305 trigger reaches CWLite glitch output.
    ChipShouter does NOT need to be armed for this test.
    """
    scope.arm()

    target.fpga_write(target.REG_CRYPT_TEXTIN, b"\x00" * 16)
    target.fpga_write(target.REG_CRYPT_GO, b"\x01")

    ret = scope.capture()

    if ret:
        print("❌ Trigger chain BROKEN — CWLite did not see trigger")
        return False
    else:
        print("✅ Trigger chain OK — glitch output will fire on next arm")
        return True

verify_trigger_chain(scope, target)

✅ Trigger chain OK — glitch output will fire on next arm


True

In [22]:
def arm_chipshouter(cs):
    """
    Safely arm ChipShouter — handles already-armed state gracefully.
    """
    try:
        cs.armed = True
        print("✅ ChipShouter armed")
    except Exception as e:
        if "armed" in str(e).lower():
            print("✅ ChipShouter already armed — continuing")
        else:
            print(f"❌ Unexpected ChipShouter error: {e}")
            raise

In [23]:
def single_em_shot(scope, target, cs, pt, key, ext_offset):
    """
    Fire ONE EM pulse and check if AES output changed.
    Returns: (ciphertext_bytes, status_string)
    """
    # ── Get reference CT first ────────────────────────────────
    target.fpga_write(target.REG_CRYPT_KEY, key)
    target.fpga_write(target.REG_CRYPT_TEXTIN, pt)
    target.fpga_write(target.REG_CRYPT_GO, b"\x01")
    time.sleep(0.01)
    ref_ct = bytes(target.fpga_read(target.REG_CRYPT_CIPHEROUT, 16))

    # ── Configure glitch offset ───────────────────────────────
    scope.glitch.ext_offset = ext_offset

    # ── Arm ChipShouter safely ────────────────────────────────
    try:
        cs.armed = True
    except Exception as e:
        if "armed" in str(e).lower():
            pass   # already armed — that's fine
        else:
            raise

    # ── Arm scope ─────────────────────────────────────────────
    scope.arm()

    # ── Fire AES → triggers full chain ───────────────────────
    target.fpga_write(target.REG_CRYPT_TEXTIN, pt)
    target.fpga_write(target.REG_CRYPT_GO, b"\x01")

    ret = scope.capture()
    if ret:
        return None, "timeout"

    ct = bytes(target.fpga_read(target.REG_CRYPT_CIPHEROUT, 16))

    # ── Classify result ───────────────────────────────────────
    if ct == ref_ct:
        return ct, "normal"
    elif ct == bytes(16):
        return ct, "zero_out"
    else:
        return ct, "FAULTED ✅"


# ── Run it ────────────────────────────────────────────────────
KEY = bytes.fromhex("2b7e151628aed2a6abf7158809cf4f3c")
PT  = bytes.fromhex("5c692f9103b2302914d7e555e4dcee49")

ct, status = single_em_shot(scope, target, cs, PT, KEY, ext_offset=0)
print(f"Status : {status}")
print(f"CT     : {ct.hex() if ct else 'None'}")

Status : normal
CT     : 06f36a65e8a99ff8907b2e5e5ddd77de


In [24]:
def verify_chipshouter_trigger(scope, target, cs, n_shots=5):
    """
    Run AES N times and check if ChipShouter shot counter increments.
    If counter goes up → trigger is reaching ChipShouter.
    """

    # ── Read shot counter BEFORE ──────────────────────────────
    shots_before = cs.shot_count
    print(f"Shot count BEFORE : {shots_before}")

    for i in range(n_shots):
        # Arm safely
        try:
            cs.armed = True
        except Exception as e:
            if "armed" in str(e).lower():
                pass
            else:
                raise

        scope.arm()
        target.fpga_write(target.REG_CRYPT_TEXTIN, b"\x00" * 16)
        target.fpga_write(target.REG_CRYPT_GO, b"\x01")
        scope.capture()

    # ── Read shot counter AFTER ───────────────────────────────
    shots_after = cs.shot_count
    print(f"Shot count AFTER  : {shots_after}")
    print(f"Shots fired       : {shots_after - shots_before}")

    if shots_after > shots_before:
        print("✅ Trigger confirmed — ChipShouter is receiving signal!")
    else:
        print("❌ Trigger NOT reaching ChipShouter — check wiring")

verify_chipshouter_trigger(scope, target, cs, n_shots=5)

AttributeError: 'ChipSHOUTER' object has no attribute 'shot_count'

In [2]:
import chipshouter
print(dir(chipshouter))

['ChipSHOUTER', '__all__', '__builtins__', '__cached__', '__doc__', '__file__', '__loader__', '__name__', '__package__', '__path__', '__spec__', 'chipshouter', 'com_tools', 'console']


In [4]:
from chipshouter import ChipSHOUTER

cs = ChipSHOUTER("/dev/ttyUSB0")   # adjust port (Windows: "COM3", etc.)
cs.voltage = 150        # starting pulse voltage in Volts (start LOW, ~150V)
cs.pulse.width = 100    # pulse width in nanoseconds — start at 100ns
cs.armed = True

In [13]:
import time
import random
from chipshouter import ChipSHOUTER
from chipshouter.com_tools import Reset_Exception

# Initialize connection
cs = ChipSHOUTER('/dev/ttyUSB0')

def setup_device(): # Added parentheses here
    # Set any defaults you want
    cs.pulse.repeat = 1

    # Arm - add some delay for next stuff
    cs.armed = 1
    time.sleep(0.5)

# Run initial setup
setup_device()

while True:
    try:
        # Note: Ensure random is imported
        cs.voltage = random.randint(200, 500)
        cs.pulse = 1
        time.sleep(0.1) # Small delay to prevent flooding the buffer
    except Reset_Exception:
        print("Device rebooted!")
        time.sleep(5) 
        setup_device()
    except Exception as e:
        print(f"An unexpected error occurred: {e}")
        break

KeyboardInterrupt: 

In [10]:
ls /dev/ttyUSB* /dev/ttyACM*

/dev/ttyACM0  /dev/ttyUSB0


In [12]:
ls /dev/ttyACM*

/dev/ttyACM0
